# CEDAR Query API - Example Search Scenarios and Queries

The Cancer Epitope Database and Analysis Resource (CEDAR) API (Application Programming Interface) enables users to perform queries and retrieve CEDAR data programmatically within their preferred environment. This Python Notebook Series illustrates basic usage examples of the CEDAR API using Python.

The CEDAR API is built on a PostgREST platform, providing transparent access to the tables on the backend. For more information on the expressive syntax of PostgREST, refer to this [document](https://postgrest.org/en/stable/api.html#). The CEDAR data can be queried through search and export endpoints described in the [CEDAR API documentation](https://discuss.iedb.org/t/application-programming-interface-api/220) and the interactive [Swagger documentation](https://cedar-api.iedb.org/docs/swagger/).

These example search scenarios and queries are derived from this [Book Chapter](https://doi.org/10.1007/978-1-0716-4566-6_3), which describes how to perform different queries to the [CEDAR homepage](https://cedar.iedb.org). This notebook reproduces the same queries using the CEDAR API.

## Research scenario IV

Neoantigen vaccines can elicit strong and durable antitumor immune responses when paired with immune-checkpoint blockade therapy. The user wants to investigate which neoantigens have been successfully used for vaccinating cancer patients.

### Approach

To retrieve this data, we will execute the following steps:
1. **Search epitopes** that have been used in the context of therapeutic vaccines
2. **Retrieve epitope information**

Each step requires a request to a different interconnected endpoint. For more information, refer to the [PostgREST resource embedding documentation](https://docs.postgrest.org/en/v12/references/api/resource_embedding.html#resource-embedding).

### Setup

First, import the required modules and define a function to make CEDAR API requests. This function takes the endpoint, the search parameters, and the base URI (Unified Resource Identifier) as inputs. Based on these, it constructs the URL (Unified Resource Locator) to interact with the data and resources specified. Then, it performs the API request. Since the API has a maximum limit of 10,000 rows per request, the function iterates through results, retrieving data in chunks. Finally, it parses the response and returns a pandas DataFrame. For more information, refer to [pandas documentation](https://pandas.pydata.org/).

In [ ]:
import os
import requests
import pandas as pd
import io
import time

pd.set_option('display.max_columns', 100)

def API_query(endpoint, query_params, base_uri='https://cedar-api.iedb.org/'):
    """
    Execute a query against the CEDAR API with pagination support.

    Parameters:
    -----------
    endpoint : str
        The API endpoint to query
    query_params : dict
        Dictionary of query parameters
    base_uri : str
        Base URI for the CEDAR API

    Returns:
    --------
    pd.DataFrame
        Combined results from all paginated requests
    """

    url = os.path.join(base_uri, endpoint)
    df = pd.DataFrame()

    # set the offset to 0
    query_params['offset'] = 0

    # loop through the pages of results
    # API only allows pulling 10,000 entries at a time
    while(True):
        print('Fetching offset: %i' % query_params['offset'])
        r = requests.get(
            url,
            params=query_params,
            headers={'accept': 'text/csv', 'Prefer': 'count=exact'}
        )

        try:
            # Parse CSV response and append to existing DataFrame
            df = pd.concat([df, pd.read_csv(io.StringIO(r.content.decode('utf-8')))])
            query_params['offset'] += 10000
        except pd.errors.EmptyDataError:
            # No more data available
            break

        # Rate limiting: pause between requests to not overload the server
        time.sleep(1)

    return df


### Step 1: Search Epitopes

In this example, we are searching for **neoepitopes** used for **vaccination**.

The CEDAR API provides two types of endpoints, the search and export. The search endpoints contain fields with information that facilitate to programatically identify and/or filter the data of interest, while the export endpoints organize the data in a user-friendly format, matching the structure and naming conventions of the custom exports on the CEDAR website.

To perform our query, we will first use the `epitope_search` endpoint. Let's examine its structure and available fields.

In [ ]:
cedar_url='https://cedar-api.iedb.org/'
endpoint='epitope_search'

table = pd.json_normalize(requests.get(os.path.join(cedar_url, endpoint + '?limit=3')).json())
display(table)

,structure_id,structure_iri,structure_descriptions,curated_source_antigens,structure_type,linear_sequence,e_modification,linear_sequence_length,iedb_assay_ids,iedb_assay_iris,...,viral_antigen_bool,germline_antigen_bool,other_antigen_bool,disease_stages,naturally_occuring_disease_bool,animal_model_of_cancer_bool,all_vaccination_bool,prophylactic_vaccination_bool,therapeutic_vaccination_bool,mutation
0,657187,IEDB_EPITOPE:657187,[DDDIAALVVDNGSG],"[{'accession': '119607748', 'name': 'hCG15971,...",Linear peptide,DDDIAALVVDNGSG,None,14.0,"[3390839, 3390840, 3572135, 4077251, 4077252, ...","[IEDB_ASSAY:11176248, IEDB_ASSAY:16529572, IED...",...,0,0,1,None,1,0,0,0,0,None
1,655612,IEDB_EPITOPE:655612,[ARILQGQGY],"[{'accession': 'EDM01270.1', 'name': 'rCG41499...",Linear peptide,ARILQGQGY,None,9.0,"[3386908, 3386909, 4075399]","[IEDB_ASSAY:3386908, IEDB_ASSAY:3386909, IEDB_...",...,0,0,1,None,0,0,0,0,0,None
2,655613,IEDB_EPITOPE:655613,[ARILQGQGYVTGL],"[{'accession': 'NP_001041350.1', 'name': 'aryl...",Linear peptide,ARILQGQGYVTGL,None,13.0,"[3386910, 3386911]","[IEDB_ASSAY:3386910, IEDB_ASSAY:3386911]",...,0,0,1,[Acute/Recent onset],0,0,0,0,0,None
3,655628,IEDB_EPITOPE:655628,[ARIQEFFRR],"[{'accession': 'AAF67474.1', 'name': 'PC326 pr...",Linear peptide,ARIQEFFRR,None,9.0,"[3386956, 3386957, 4075424]","[IEDB_ASSAY:3386956, IEDB_ASSAY:3386957, IEDB_...",...,0,0,1,None,0,0,0,0,0,None
4,655658,IEDB_EPITOPE:655658,[ARIVTDRETGSSK],"[{'accession': 'EDL75577.1', 'name': 'nucleoli...",Linear peptide,ARIVTDRETGSSK,None,13.0,"[3387048, 3387049]","[IEDB_ASSAY:3387048, IEDB_ASSAY:3387049]",...,0,0,1,[Acute/Recent onset],0,0,0,0,0,None


Next, we define the search parameters, which consist of three components:

- The **query filters** that we want to impose on the data.

  To filter viral antigens, we will use the `neoantigen_bool` field, specific to the search endpoint, whose value is 1 if the antigen is a neoantigen, and 0 otherwise.

  To filter by epitopes studied in humans, we will use the field `host_organism_iris`. CEDAR is connected to external resources using IRIs (Internationalized Resource Identifiers) that point to other databases, such as the [NCBI Taxonomy](https://www.ncbi.nlm.nih.gov/taxonomy), allowing users to filter the data using external stable identifiers. The NCBI Taxonomy ID of humans (*homo sapiens*) is [9606](https://www.ncbi.nlm.nih.gov/Taxonomy/Browser/wwwtax.cgi?mode=Info&id=9606&lvl=3&lin=f&keep=1&srchmode=1&unlock).

  To filter by epitopes used in the context of vaccination, we can use two different boolean fields, `all_vaccination_bool`, and `therapeutic_vaccination_bool`. The first one will retrieve all the epitopes used in any type of vaccine, whereas the second will retrieve a subset of epitopes used in therapeutic vaccines. This is particularly relevant in cancer vaccines, as most of them are administered as therapeutic agents.

  Also, we want to retrieve only the positive assays, so use the `qualitative_measures` field to only retrieve positive assays.

  To apply these filters, we utilize operators such as `eq`, `cs`, and `in`, which allow us to perform flexible queries and refine the search. For more information on the operators' syntax, refer to this [documentation](https://docs.postgrest.org/en/v12/references/api/tables_views.html#operators).

- The **pagination parameters** required to retrieve the results in chunks. The order parameter is a field used to sort and paginate the data so that each retrieved chunk contains different consecutive rows. The offset parameter indicates the index of the first element of each data chunk.

- The final **column selection**. In this case, we will keep the assay IDs of the `structure_id` field. These will be used for subsequent queries to other endpoints.

In [ ]:
search_params={
    # Query filters
    'neoantigen_bool': 'eq.1',
    'host_organism_iri': 'eq.NCBITaxon:9606',
    'therapeutic_vaccination_bool': 'eq.1',
    'qualitative_measures':'ov.{Positive,Positive-Low,Positive-Intermediate,Positive-High}',

    # Pagination
    'order': 'structure_id',
    'offset': 0,

    # Column selection
    'select': 'structure_id',
}

# Execute epitope search
epitope_search_df = API_query(endpoint, search_params)

print(f"Retrieved {len(epitope_search_df)} epitope records")
epitope_search_df.head()

Fetching offset: 0
Fetching offset: 10000
Retrieved 648 epitope records


,structure_id
0,3147
1,14672
2,23214
3,28958
4,37113


### Step 2: Retrieve epitope information

With the epitope IDs of interest, we map their complete information by querying the ``epitope_export`` endpoint. This requires formatting the `structure_id`, and including them in the search parameters.

In [ ]:
epitope_ids = ','.join(map(str,set(epitope_search_df['structure_id'].to_list())))
endpoint='epitope_export'

search_params = {
    # Query filter
    'structure_id': 'in.(' + epitope_ids + ')',

    # Pagination
    'order': 'structure_id',
    'offset': 0,
}

# Execute final query
epitope_export_df = API_query(endpoint, search_params)

print(f"Retrieved {len(epitope_export_df)} epitopes")
epitope_export_df.head()

Fetching offset: 0
Fetching offset: 10000
Retrieved 731 epitopes


,structure_id,epitope_id__cedar_iri,epitope__object_type,epitope__name,epitope__modified_residues,epitope__modifications,epitope__starting_position,epitope__ending_position,epitope__iri,epitope__synonyms,...,related_object__iri,related_object__synonyms,related_object__source_molecule,related_object__source_molecule_iri,related_object__molecule_parent,related_object__molecule_parent_iri,related_object__source_organism,related_object__source_organism_iri,related_object__species,related_object__species_iri
0,3147,https://cedar.iedb.org/epitope/3147,Linear peptide,AMLGTHTMEV,NaN,NaN,177.0,186.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3147,https://cedar.iedb.org/epitope/3147,Linear peptide,AMLGTHTMEV,NaN,NaN,177.0,186.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3147,https://cedar.iedb.org/epitope/3147,Linear peptide,AMLGTHTMEV,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,"ME20-M, Melanocyte protein Pmel 17, Melanocyte...",Melanocyte protein PMEL,https://www.uniprot.org/uniprot/P40967.2,Melanocyte protein PMEL,http://www.uniprot.org/uniprot/P40967,Homo sapiens (human),http://purl.obolibrary.org/obo/NCBITaxon_9606,Homo sapiens,http://purl.obolibrary.org/obo/NCBITaxon_9606
3,3147,https://cedar.iedb.org/epitope/3147,Linear peptide,AMLGTHTMEV,NaN,NaN,184.0,193.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,14672,https://cedar.iedb.org/epitope/14672,Linear peptide,EVDPIGHLY,NaN,NaN,168.0,176.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Results

The final DataFrame contains all the information about Immunogenic neoepitopes used for therapeutic vaccination in humans. This programmatic approach demonstrates the CEDAR API's capability for automated data retrieval and analysis.

## References

Koşaloğlu-Yalçın, Z. et al. (2025) *Using the Cancer Epitope Database and Analysis Resource (CEDAR)*. Alexander Krasnitz and Pascal Belleau (Eds.), *Cancer Bioinformatics, Methods in Molecular Biology*, vol. 2932. [https://doi.org/10.1007/978-1-0716-4566-6_3 ](https://doi.org/10.1007/978-1-0716-4566-6_3)

Koşaloğlu-Yalçın, Z. et al. (2023) *The cancer epitope database and analysis resource (CEDAR).* Nucleic acids research 51, no. D1 (2023): D845-D852. [https://doi.org/10.1093/nar/gkac902 ](https://doi.org/10.1093/nar/gkac902)